# Application code sample — Cohort quality control

**Project:** MIMIC-IV recorded 30-day hospital readmission analysis.  
**Applicant:** Jessica Xu.

## Contribution and AI assistance
Python code was generated with AI assistance. My project work focuses on reviewing research definitions, investigating data inconsistencies, and documenting analytical decisions. This sample demonstrates the resulting analytical workflow; it is not a claim of independently authored Python code.

## Running this sample
Requires Python 3 and pandas, plus Jupyter/IPython for notebook display. Obtain authorized access to MIMIC-IV separately; no source data are included. Place admissions.csv and patients.csv in data/raw relative to the notebook working directory, or change DATA_DIR in the setup cell. Run the cohort notebook first, then the labeling notebook, using the same working directory. Outputs are written under sample_outputs; existing files there will be replaced when export cells run.

## Review status and privacy
These are reviewed Steps 2 and 3 from an ongoing revision of a previously completed analysis. Earlier model results use a different cohort and are not results of this revised pipeline. Selected aggregate outputs are preserved verbatim from the applicant-provided notebook execution. Patient-level example outputs, local-path outputs, and execution counts have been removed. The aggregate results are historical saved outputs, not results recomputed during application preparation. Setup paths were adapted for portability; the analysis was not rerun after that change. Running the notebooks requires restricted data and may produce sensitive local outputs, which should not be shared.

## Scope
Validates timestamps, compares mortality indicators, applies sequential eligibility rules, and exports a cohort with validation checks.


# Step 2 — Cohort Definition and Quality Control

## Purpose

This notebook defines the eligible admission-level sample for the
30-day hospital readmission analysis.

The existing analysis starts with the raw admissions table, requires
nonmissing admission and discharge timestamps, excludes records
indicating in-hospital death, excludes hospice discharges, and applies
an adult filter using anchor_age when that field is available.

This review verifies the implemented criteria, documents exclusions,
and checks the resulting cohort before label construction.

## Unit of Observation

Each row represents one eligible hospital admission, identified by
`hadm_id`.

A patient, identified by `subject_id`, may contribute multiple admissions.

**One-line memory:**
Patients = one row per person.
Admissions = one row per hospital stay.

## Inputs and Output

**Inputs:**
- Admissions table: hospital stays and admission-level information.
- Patients table: patient-level information used for the age filter.
- Shared `config.py`: source paths and processed-data directory.

**Output:**
- `data/processed/cohort_step2.csv`
- An eligible admission-level cohort for Step 3 label construction.

## Cohort Rules in the Existing Analysis

| Criterion | Existing implementation | Purpose |
|---|---|---|
| Admission and discharge times | Convert timestamps and exclude missing values | Retain records that support time-based labeling |
| In-hospital mortality | Retain records with `hospital_expire_flag == 0` and discharge disposition other than `DIED` | Exclude admissions indicating in-hospital death |
| Hospice discharge | Identify and exclude hospice-related discharge dispositions | Apply the study's hospice exclusion |
| Adult population | Apply `anchor_age >= 18` | Apply the existing adult eligibility rule |

These rules describe the current implementation. Timestamp ordering,
disposition handling, and the use of anchor_age require verification
during this review.

## Pipeline Design Principle — From Cohort to Final Model

The analysis separates the eligible sample, outcome, predictors,
and model. Each step creates a different part of the final analysis.

### Step 2 — Define the Analysis Sample (S)

**Goal:** Determine which hospital admissions are eligible.

**Mathematical view:**

$$
S = \{\text{admissions satisfying inclusion/exclusion criteria}\}
$$

**Properties:**
- S is the eligible admission-level sample.
- One patient may contribute multiple admissions.
- Eligibility follows study criteria.
- Eligibility should not depend on individual predictor availability.
- Outcome labels and predictive features are constructed later.

**Output:**

$$
S
$$

### Step 3 — Define the Outcome Variable (Y)

**Goal:** Define what the model predicts.

**Mathematical view:**

$$
Y = \operatorname{readmission}_{30d}(S)
$$

**Properties:**
- Each eligible admission receives a readmission outcome.
- Y may use future information, such as a subsequent admission.
- The qualifying-readmission definition and follow-up must be verified.
- Future information used for Y must not enter the predictors.

**Output:**

$$
(S, Y)
$$

### Step 4 — Construct Predictive Features (X)

**Goal:** Define what information the model can use at discharge.

**Mathematical view:**

$$
X_{\text{Tier 1}} = f_{\text{Tier 1}}(S)
$$

$$
X_{\text{Tier 2}} = f_{\text{Tier 2}}(S)
$$

$$
X_{\text{Tier 3}} = f_{\text{Tier 3}}(S)
$$

$$
X = [X_{\text{Tier 1}} \mid X_{\text{Tier 2}} \mid X_{\text{Tier 3}}]
$$

**Feature groups:**
- Tier 1: demographic, administrative, and prior-utilization features.
- Tier 2: structured clinical features.
- Tier 3: medication, laboratory, and vital-sign features.

**Critical rules:**
- Use only information available by the discharge prediction point.
- Do not use future admissions or outcome-derived variables as predictors.
- Handle missing predictors through appropriate preprocessing and
  missingness indicators.
- Changing feature richness should not silently change S.

**Output:**

$$
(S, Y, X)
$$

### Step 5 — Create Training, Validation, and Test Sets

**Goal:** Separate model development from final evaluation.

**Mathematical view:**

$$
(S, X, Y)
\longrightarrow
\begin{cases}
(X_{\text{train}}, Y_{\text{train}}) \\
(X_{\text{validation}}, Y_{\text{validation}}) \\
(X_{\text{test}}, Y_{\text{test}})
\end{cases}
$$

**Critical rules:**
- Split by patient (`subject_id`), rather than by admission.
- Keep all admissions from one patient in the same set.
- Check that patient groups do not overlap between sets.
- Preserve alignment between predictor rows and outcome labels.
- Remove identifiers and leakage-prone fields from model predictors.

**Why this matters:**
Patient overlap could make evaluation overly optimistic because
the model has already encountered information from the same person.

### Step 6 — Train, Select, and Evaluate Models

**Goal:** Learn relationships between predictors and readmission.

**Mathematical view:**

$$
\hat{g}_m =
\operatorname{Train}_m(X_{\text{train}}, Y_{\text{train}})
$$

Here, m represents a candidate model:
Logistic Regression, Random Forest, or XGBoost.

The fitted model produces predicted readmission probabilities:

$$
\hat{p}_i = \hat{g}_m(x_i)
$$

A selected threshold t converts probabilities into binary predictions:

$$
\hat{y}_i = \mathbf{1}(\hat{p}_i \geq t)
$$

**Critical rules:**
- Fit preprocessing and candidate models using training data only.
- Use validation data for model comparison and threshold selection.
- Evaluate the selected pipeline on the held-out test set.
- Report discrimination and threshold-dependent performance.

**One-line memory:**
Training learns; validation guides choices; testing evaluates
the selected approach.

### Step 7 — Interpret the Model

**Goal:** Understand which measured features contribute to predictions.

**Conceptual view:**

$$
\text{Model explanation}
=
\operatorname{Interpret}(\hat{g}, X)
$$

**Critical rules:**
- Examine feature importance and SHAP contributions.
- Explain the same fitted model evaluated in Step 6.
- Identify and separately evaluate any refitted interpretation model.
- Treat feature contributions as explanations of model behavior,
  rather than evidence of causal effects.

## Core Conceptual Equation

$$
S
\longrightarrow
(S, Y)
\longrightarrow
(S, Y, X)
\longrightarrow
\text{Patient-level splits}
\longrightarrow
\hat{p} = \hat{g}(X)
$$

**One-line memory:**

Step 2 defines WHO is eligible.
Step 3 defines WHAT happened.
Step 4 defines WHAT the model can see.
Step 5 separates patients.
Step 6 predicts and evaluates.
Step 7 explains.

## Central Design Principle

Sample (S), outcome (Y), predictors (X), and model (g)
are conceptually distinct.

Changing feature richness or model choice should not silently
redefine the eligible sample.

Any necessary revision to eligibility must be documented and
carried through label construction, feature engineering,
splitting, and model evaluation.

## Step 2 Validation Goals

- Verify required source columns and identifier integrity.
- Check timestamp parsing and admission/discharge ordering.
- Examine agreement between mortality indicators.
- Verify hospice-discharge classification.
- Verify the age rule and handling of missing age information.
- Record admissions removed at each filtering stage.
- Report both eligible admissions and unique patients.
- Confirm that final admission identifiers remain unique.
- Display aggregate results without individual patient records.

## Boundary Between Cohort Selection and Label Construction

Step 2 determines which admissions are eligible as prediction
observations.

An admission excluded as a prediction observation may still be
relevant as a subsequent readmission event for another admission.
Step 3 must therefore verify which source records are searched
when identifying outcomes.

## 2.1 — Load Project Configuration

Import the required libraries and locate the shared config.py.
Retrieve source-file paths and the processed-data directory explicitly
so the notebook's dependencies are clear.

In [ ]:
from pathlib import Path
import pandas as pd

# Set this to your authorized local MIMIC-IV hospital data directory.
DATA_DIR = Path("data/raw")
ADMISSIONS_FILE = DATA_DIR / "admissions.csv"
PATIENTS_FILE = DATA_DIR / "patients.csv"
PROCESSED_DIR = Path("sample_outputs")

# Fail clearly when required local inputs are absent.
for input_path in [ADMISSIONS_FILE, PATIENTS_FILE]:
    if not input_path.is_file():
        raise FileNotFoundError(f"Required local input missing: {input_path}")


## 2.2 — Load and Validate Source Tables

Load the admissions and patients tables using the configured paths.
Confirm that the columns needed for cohort construction are present
before applying eligibility rules.

Create a separate working copy of admissions and record the initial
numbers of admissions and unique patients. These counts provide the
baseline for documenting subsequent exclusions.

No eligibility filters are applied in this section.

In [ ]:
# Verify that both source paths point to existing files.
for source_file in [ADMISSIONS_FILE, PATIENTS_FILE]:
    # Stop with the failed path if a source file is unavailable.
    if not source_file.is_file():
        raise FileNotFoundError(f"Source file not found: {source_file}")

# Load hospital admission records.
admissions = pd.read_csv(ADMISSIONS_FILE)

# Load patient records.
patients = pd.read_csv(PATIENTS_FILE)

# List columns required for the planned cohort rules.
required_columns = {
    "admissions": {
        "subject_id",
        "hadm_id",
        "admittime",
        "dischtime",
        "hospital_expire_flag",
        "discharge_location",
    },
    "patients": {"subject_id", "anchor_age"},
}
# Requiring anchor_age prevents silently skipping the adult filter.

# Associate each table name with its DataFrame.
source_tables = {
    "admissions": admissions,
    "patients": patients,
}

# Check the required columns in each table.
for table_name, table in source_tables.items():
    # Identify required names absent from the loaded table.
    missing_columns = required_columns[table_name] - set(table.columns)

    # Stop before filtering if required information is unavailable.
    if missing_columns:
        raise ValueError(
            f"{table_name} is missing required columns: "
            f"{sorted(missing_columns)}"
        )

# Create the working cohort without modifying the admissions DataFrame.
cohort = admissions.copy()

# Record admission rows before exclusions.
n_initial = len(cohort)

# Record distinct patients before exclusions.
n_initial_patients = cohort["subject_id"].nunique()

# Display aggregate starting counts without individual records.
display(pd.DataFrame({
    "stage": ["Before exclusions"],
    "admissions": [n_initial],
    "unique_patients": [n_initial_patients],
}))

,stage,admissions,unique_patients
0,Before exclusions,546028,223452


## 2.3 — Validate Admission and Discharge Times

Convert source timestamps into datetime values. Count originally
missing timestamps separately from values that fail conversion.

Exclude admissions without both usable timestamps, following the
existing eligibility rule.

Check whether discharge precedes admission. If such records exist,
stop for investigation before applying further exclusions.
Equal admission and discharge times are retained in this check.

In [ ]:
# Store timestamp diagnostics without displaying individual records.
timestamp_checks = []

# Convert each required time column.
for column in ["admittime", "dischtime"]:
    # Use the original source values to distinguish missingness from errors.
    source_values = admissions[column]

    # Convert readable values; unreadable values become NaT.
    parsed_values = pd.to_datetime(source_values, errors="coerce")

    # Count values originally missing in the source.
    source_missing = int(source_values.isna().sum())

    # Count nonmissing source values that could not be converted.
    parsing_failures = int(
        (source_values.notna() & parsed_values.isna()).sum()
    )

    # Record aggregate diagnostics for this column.
    timestamp_checks.append({
        "column": column,
        "source_missing": source_missing,
        "parsing_failures": parsing_failures,
    })

    # Add converted timestamps to the working cohort.
    # Matching row indexes align these values with the correct admissions.
    cohort[column] = parsed_values

# Display missingness and conversion results.
display(pd.DataFrame(timestamp_checks))

# Identify admissions with both usable timestamps.
usable_times = (
    cohort["admittime"].notna()
    & cohort["dischtime"].notna()
)

# Count excluded rows once, even if both timestamps are missing.
time_removed = int((~usable_times).sum())

# Retain admissions with both usable timestamps.
cohort = cohort.loc[usable_times].copy()

# Preserve the count used by the later filtering summary.
n_after_time = len(cohort)

# Identify stays whose discharge precedes admission.
invalid_order = cohort["dischtime"] < cohort["admittime"]

# Report the number of chronological inconsistencies.
invalid_order_count = int(invalid_order.sum())

# Display aggregate results.
print("Admissions removed for unusable timestamps:", time_removed)
print("Admissions remaining:", n_after_time)
print("Discharge before admission:", invalid_order_count)

# Report reversed timestamps without stopping execution.
# The following cells investigate these records and then exclude them.
if invalid_order_count > 0:
    print(
        "Reversed timestamps detected. "
        "These records are investigated and excluded in the following cells."
    )

,column,source_missing,parsing_failures
0,admittime,0,0
1,dischtime,0,0


Admissions removed for unusable timestamps: 0
Admissions remaining: 546028
Discharge before admission: 175
Reversed timestamps detected. These records are investigated and excluded in the following cells.


### 2.3.1 — Investigate Reversed Admission and Discharge Times

The timestamp validation identified 175 admissions whose recorded
discharge time preceded admission time.

This diagnostic examines:
- The magnitude of the negative hospital-stay durations.
- Whether discrepancies are within one hour, within 24 hours, or longer.
- The admission types represented among these records.

Only aggregate statistics are displayed. These records remain in the
working cohort during investigation.

#### Findings

- 49 admissions had discrepancies of up to one hour.
- 126 had discrepancies greater than one hour but within 24 hours.
- None had discrepancies greater than 24 hours.
- The median discrepancy was approximately 6.87 hours.
- The largest discrepancy was approximately 22.68 hours.

These results establish a chronological inconsistency, but do not
identify its cause. Admission type alone cannot explain the discrepancy.

In [ ]:
# Select records with discharge earlier than admission.
# Keep individual records internal; display only aggregate results.
time_anomalies = cohort.loc[invalid_order].copy()

# Calculate hospital-stay duration in hours.
duration_hours = (
    time_anomalies["dischtime"] - time_anomalies["admittime"]
).dt.total_seconds() / 3600

# Summarize the magnitude of the negative durations.
display(duration_hours.describe().to_frame("negative_duration_hours"))

# Group anomalies by how far discharge precedes admission.
display(pd.DataFrame({
    "time_difference": [
        "Up to 1 hour",
        "More than 1 hour, up to 24 hours",
        "More than 24 hours",
    ],
    "admissions": [
        int((duration_hours >= -1).sum()),
        int(((duration_hours < -1) & (duration_hours >= -24)).sum()),
        int((duration_hours < -24).sum()),
    ],
}))

# Summarize affected admission types without patient identifiers.
display(
    time_anomalies["admission_type"]
    .value_counts(dropna=False)
    .to_frame("admissions")
)

,negative_duration_hours
count,175.000000
mean,-8.060857
std,7.189809
min,-22.683333
25%,-14.366667
50%,-6.866667
75%,-0.791667
max,-0.016667


,time_difference,admissions
0,Up to 1 hour,49
1,"More than 1 hour, up to 24 hours",126
2,More than 24 hours,0


,admissions
admission_type,
EW EMER.,78
EU OBSERVATION,56
URGENT,19
OBSERVATION ADMIT,8
DIRECT OBSERVATION,6
DIRECT EMER.,5
SURGICAL SAME DAY ADMISSION,2
AMBULATORY OBSERVATION,1


### 2.3.2 — Check Overlap with Other Exclusion Criteria

Check whether the 175 admissions with reversed timestamps also have
mortality or hospice indicators.

This diagnostic counts:
- Records indicating death through hospital_expire_flag or a DIED
  discharge disposition.
- Records with a hospice-related discharge disposition.
- Records with neither indicator.

#### Why This Matters

Some timestamp anomalies may already meet another exclusion criterion.
Checking overlap helps distinguish the total number of anomalies from
the additional admissions a chronological-validity rule would exclude.

Death and hospice counts may overlap and should not be added together.
Having neither indicator does not establish eligibility under every
other study criterion.


In [ ]:
# Standardize discharge descriptions for this diagnostic only.
disposition = (
    time_anomalies["discharge_location"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Identify records where either mortality field indicates death.
death_indicated = (
    time_anomalies["hospital_expire_flag"].eq(1)
    | disposition.eq("DIED").fillna(False)
)

# Identify hospice-related discharge descriptions.
hospice_indicated = disposition.str.contains("HOSPICE", na=False)

# Count overlapping exclusion indicators without displaying records.
display(pd.DataFrame({
    "category": [
        "Death indicated",
        "Hospice indicated",
        "Neither death nor hospice indicated",
    ],
    "admissions": [
        int(death_indicated.sum()),
        int(hospice_indicated.sum()),
        int((~death_indicated & ~hospice_indicated).sum()),
    ],
}))
# Death and hospice counts may overlap; they are not additive.

,category,admissions
0,Death indicated,101
1,Hospice indicated,0
2,Neither death nor hospice indicated,74


#### Interpretation — Overlap with Other Exclusions

Among the 175 admissions with discharge earlier than admission,
101 also had an in-hospital death indicator, none had a hospice
indicator, and 74 had neither indicator.

The 101 death-indicated admissions already meet the mortality
exclusion criterion. A separate chronological-validity exclusion
would remove 74 additional admissions beyond the death and hospice
criteria, before considering age eligibility.

The cause of the reversed timestamps remains unresolved.
No timestamps were corrected and no admissions were removed
during this diagnostic.

MIMIC-IV documentation notes that separate organ-donor accounts
may have very short or negative lengths of stay. This is a possible
explanation for some death-associated timestamp anomalies, but donor
status was not confirmed in this analysis.

Other reversed-time records remain unexplained. These timestamps
were not automatically corrected.

### 2.3.3 — Examine Equal Admission and Discharge Times

Identify admissions whose recorded admission and discharge timestamps
are identical. These records have a recorded length of stay of zero.

This diagnostic counts equal-time admissions and summarizes their
admission types without displaying individual patient records.

#### Why This Matters

Equal timestamps are not chronologically reversed, but they do not
by themselves confirm a valid hospital stay. They may reflect a brief
encounter or a recording issue; the cause requires investigation.

#### Inclusion Rule

Retain equal-time admissions provisionally under the chronological rule:

`dischtime >= admittime`

These admissions must still satisfy the mortality, hospice, age,
and other eligibility criteria. Do not adjust their timestamps or
exclude them solely because the recorded duration is zero.

#### Implication for Label Construction

Step 3 can calculate the interval from discharge to a subsequent
admission. A subsequent encounter beginning at exactly the same time
requires review for transfers or encounter duplication before being
classified as a readmission.

#### Interpretation Pending

Report the number and admission types of equal-time records after
running the diagnostic. No admissions are removed by this check.

In [ ]:
# Identify admissions with identical admission and discharge timestamps.
equal_times = cohort["admittime"].eq(cohort["dischtime"])

# Count these admissions without displaying individual records.
print("Admissions with equal timestamps:", int(equal_times.sum()))

# Examine the admission types represented.
display(
    cohort.loc[equal_times, "admission_type"]
    .value_counts(dropna=False)
    .to_frame("admissions")
)

Admissions with equal timestamps: 5


,admissions
admission_type,
EU OBSERVATION,3
DIRECT EMER.,1
SURGICAL SAME DAY ADMISSION,1


#### Interpretation — Equal Admission and Discharge Times

Five admissions had identical admission and discharge timestamps:
three EU OBSERVATION, one DIRECT EMER., and one SURGICAL SAME DAY
ADMISSION.

These records have a recorded length of stay of zero. Their admission
types do not establish whether the timestamps reflect brief encounters
or recording issues.

No admissions were excluded solely because their timestamps were equal.
They remain subject to mortality, hospice, age, and other eligibility
criteria.

In [ ]:
# Select the five admissions with equal admission and discharge times.
equal_time_records = cohort.loc[equal_times].copy()

# Standardize discharge descriptions for consistent comparisons.
equal_disposition = (
    equal_time_records["discharge_location"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Identify death using either the mortality flag or discharge description.
equal_death = (
    equal_time_records["hospital_expire_flag"].eq(1)
    | equal_disposition.eq("DIED").fillna(False)
)

# Identify discharge descriptions containing "HOSPICE".
equal_hospice = equal_disposition.str.contains("HOSPICE", na=False)

# Summarize eligibility without displaying individual patient records.
print("Equal-time admissions with death indicated:", int(equal_death.sum()))
print("Equal-time admissions with hospice indicated:", int(equal_hospice.sum()))
print(
    "Equal-time admissions with neither indicated:",
    int((~equal_death & ~equal_hospice).sum())
)

Equal-time admissions with death indicated: 0
Equal-time admissions with hospice indicated: 0
Equal-time admissions with neither indicated: 5


### Interpretation — Eligibility of Equal-Time Admissions

Among the five admissions with identical admission and discharge
timestamps, none indicated in-hospital death or hospice discharge.

All five were retained at the timestamp-validation stage because equal
timestamps alone were not considered sufficient evidence of an invalid
record. They remain subject to the remaining cohort eligibility criteria.

In [ ]:
# Count records before applying the time-order criterion.
n_before_time_order = len(cohort)

# Accept discharge at or after admission, including equal timestamps.
valid_time_order = cohort["dischtime"] >= cohort["admittime"]

# Record how many admissions fail this criterion before filtering.
time_order_removed = int((~valid_time_order).sum())

# Exclude reversed-time admissions from the prediction cohort.
# The original admissions DataFrame remains available for investigation.
cohort = cohort.loc[valid_time_order].copy()

# Store the remaining count for the cohort-flow summary.
n_after_time_order = len(cohort)

print("Admissions before time-order exclusion:", n_before_time_order)
print("Admissions excluded for reversed timestamps:", time_order_removed)
print("Admissions remaining:", n_after_time_order)

Admissions before time-order exclusion: 546028
Admissions excluded for reversed timestamps: 175
Admissions remaining: 545853


### Summary — Timestamp Validation and Cohort Decision

All 546,028 admissions had usable admission and discharge timestamps;
none were excluded for missing or unparseable timestamps.

We excluded 175 admissions whose discharge time preceded admission time.
These records could not provide a reliable admission-to-discharge interval
for this analysis. Their timestamps were not corrected, and the original
admissions table was preserved.

Five admissions had identical admission and discharge timestamps.
All five were retained at this stage; none indicated death or hospice
discharge. They remain subject to the remaining eligibility criteria.

**After timestamp validation, 545,853 admissions remained.**

These exclusions apply to individual admissions, not entire patients.
Subsequent exclusion counts will be calculated from the remaining cohort
to avoid counting the same admission more than once.

## 2.4 — In-Hospital Death and Hospice Exclusions

### What we are checking

We identify admissions with:
- In-hospital death, indicated by `hospital_expire_flag = 1`
  or a discharge location of `DIED`.
- Hospice discharge, indicated by a discharge location containing `HOSPICE`.

### Why we apply these criteria

Our prediction point is discharge. Admissions ending in death are excluded
because the patient cannot experience a subsequent readmission.

Hospice discharges are excluded under our study's cohort definition because
end-of-life care represents a different care context. This does not mean
hospice patients cannot be readmitted.

These criteria apply to individual admissions. We do not exclude all
admissions for a patient simply because they died during a later admission.

### How we count exclusions

Counts are calculated after timestamp validation. Admissions already
excluded for reversed timestamps are not counted again. Death and hospice
indicators are combined so each admission is removed only once.


### 2.4.1 — Check Agreement Between Mortality Fields

We compare `hospital_expire_flag` with whether `discharge_location`
is recorded as `DIED`.

Disagreement means the fields provide conflicting indications of
in-hospital death. This comparison does not establish which field is
correct or explain the cause.

For cohort eligibility, we exclude an admission when either field
indicates death. We preserve the original values rather than overwrite
one field with the other.

This diagnostic is performed before death/hospice exclusion so that
the relevant records remain available for comparison.

In [ ]:
# Standardize discharge descriptions without changing the original values.
mortality_disposition = (
    cohort["discharge_location"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Preserve missing discharge locations as a separate category.
location_status = mortality_disposition.eq("DIED").map({
    False: "Other recorded location",
    True: "DIED"
}).fillna("Missing location")

# Compare the mortality flag with the discharge-location indication.
mortality_table = pd.crosstab(
    cohort["hospital_expire_flag"].fillna("Missing flag"),
    location_status,
    margins=True,
    margins_name="Total"
)

print("Admissions included in this diagnostic:", len(cohort))
display(mortality_table)

# Count disagreements only where both fields have known values.
flag_alive = cohort["hospital_expire_flag"].eq(0)
flag_dead = cohort["hospital_expire_flag"].eq(1)
location_died = mortality_disposition.eq("DIED").fillna(False)
location_known = mortality_disposition.notna()

alive_but_died = flag_alive & location_died
dead_but_other = flag_dead & location_known & ~location_died

print("Flag = 0, location = DIED:", int(alive_but_died.sum()))
print("Flag = 1, other recorded location:", int(dead_but_other.sum()))
print(
    "Total disagreements with both fields recorded:",
    int((alive_but_died | dead_but_other).sum())
)

Admissions included in this diagnostic: 545853


discharge_location,DIED,Missing location,Other recorded location,Total
hospital_expire_flag,,,,
0,227,149592,384334,534153
1,11396,161,143,11700
Total,11623,149753,384477,545853


Flag = 0, location = DIED: 227
Flag = 1, other recorded location: 143
Total disagreements with both fields recorded: 370


### Interpretation — Mortality-Field Consistency

Among the 545,853 admissions evaluated before death/hospice exclusion:

- 11,396 had both `hospital_expire_flag = 1` and discharge location `DIED`.
- 227 had `hospital_expire_flag = 0` but discharge location `DIED`.
- 143 had `hospital_expire_flag = 1` but another recorded discharge location.
- 161 had `hospital_expire_flag = 1` and a missing discharge location.

There were **370 disagreements between recorded values** (227 + 143).
The 161 missing discharge locations were reported separately as incomplete
information. These checks do not establish which field is correct or
explain the discrepancies.

**Cohort decision:** An admission was excluded when either field indicated
in-hospital death. This identified **11,927 admissions**
(11,396 + 227 + 143 + 161).

All three groups of concern—227, 143, and 161 admissions—were therefore
covered by the death exclusion rule. No separate inconsistency exclusion
was applied, and the original field values were preserved.

The following cell combines death and hospice criteria, counting each
excluded admission only once.

### 2.4.2 — Apply Death and Hospice Exclusions

We now exclude admissions where either mortality field indicates
in-hospital death or the discharge location indicates hospice.

The criteria are combined so each admission is excluded only once.

In [ ]:
# Record the cohort size before applying death and hospice exclusions.
n_before_death_hospice = len(cohort)

# Create a standardized version of the discharge-location column.

# "disposition" means the recorded outcome or destination at discharge.
# This is a temporary Series; the original column is not changed.
disposition = (
    cohort["discharge_location"]

    # Use pandas' string type so text operations preserve missing values.
    .astype("string")

    # Remove spaces at the beginning and end, e.g., " DIED " becomes "DIED".
    .str.strip()

    # Convert text to uppercase, e.g., "Died" becomes "DIED".
    .str.upper()
)

# Create one True/False death indicator for each admission.
# True means at least one field indicates in-hospital death.
# This line identifies records; it does not remove them.
death_indicated = (
    # First condition: does the recorded mortality flag equal 1?
    # .eq(1) means "equal to 1" and is equivalent to == 1.
    cohort["hospital_expire_flag"].eq(1)

    # | means element-by-element OR:
    # an admission is flagged when either condition is True.
    |
    
    # Second condition: is the standardized discharge location "DIED"?
    # This also identifies the 227 cases whose mortality flag is 0.
    disposition.eq("DIED")

    # If discharge location is missing, this condition becomes False.
    # Missing location alone does not indicate death.
    # The first condition can still identify death when the flag is 1.
    .fillna(False)
)

# Identify hospice discharge.
hospice_indicated = disposition.str.contains("HOSPICE", na=False)

# Combine criteria so an admission meeting both is removed only once.
exclude_death_hospice = death_indicated | hospice_indicated

# Save counts before modifying the cohort.
death_count = int(death_indicated.sum())
hospice_count = int(hospice_indicated.sum())
death_hospice_overlap = int(
    (death_indicated & hospice_indicated).sum()
)
death_hospice_removed = int(exclude_death_hospice.sum())

# Retain admissions meeting neither exclusion criterion.
cohort = cohort.loc[~exclude_death_hospice].copy()

# Store the remaining count for the cohort-flow summary.
n_after_death_hospice = len(cohort)

print("Admissions before death/hospice exclusion:", n_before_death_hospice)
print("Admissions with death indicated:", death_count)
print("Admissions with hospice indicated:", hospice_count)
print("Admissions meeting both criteria:", death_hospice_overlap)
print("Unique admissions excluded:", death_hospice_removed)
print("Admissions remaining:", n_after_death_hospice)

Admissions before death/hospice exclusion: 545853
Admissions with death indicated: 11927
Admissions with hospice indicated: 5397
Admissions meeting both criteria: 22
Unique admissions excluded: 17302
Admissions remaining: 528551


### Interpretation — Death and Hospice Exclusions

Among the 545,853 admissions remaining after timestamp validation,
11,927 indicated in-hospital death and 5,397 indicated hospice discharge.
Twenty-two admissions met both criteria.

Combining these indicators excluded 17,302 unique admissions, leaving
528,551 admissions for the remaining eligibility checks.

Counts apply to individual admissions and exclude records already removed
during timestamp validation. Meeting both indicators does not by itself
establish a recording error.

## 2.5 — Age at Admission and Adult Eligibility

### What we are checking

We link each admission to the patient's `anchor_age` and `anchor_year`
to estimate age in the year of admission.

The patients table contains one row per patient, while the cohort contains
one row per admission. The merge must therefore preserve the number of
admissions.

### Why we calculate admission-specific age

`anchor_age` describes age in the patient's anchor year. It is not
necessarily the patient's age during every admission.

We calculate:

age_at_admission = anchor_age + admission_year - anchor_year

This is a year-based estimate rather than an exact age calculated from
date of birth.

### Eligibility rule

We retain admissions with an estimated age of at least 18 years.
Missing age information is investigated before filtering.

In [ ]:
# Confirm the patient fields needed to calculate age are available.
required_age_columns = {"subject_id", "anchor_age", "anchor_year"}
missing_age_columns = required_age_columns - set(patients.columns)

if missing_age_columns:
    raise ValueError(
        f"Missing patient columns: {sorted(missing_age_columns)}"
    )

# Record the admission count before adding patient information.
n_before_age_merge = len(cohort)

# Add age information.
# many_to_one requires at most one matching patient row per subject_id.
# A left merge preserves admissions even when patient information is missing.
cohort = cohort.merge(
    patients[["subject_id", "anchor_age", "anchor_year"]],
    on="subject_id",
    how="left",
    validate="many_to_one"
)

# Confirm that the merge did not add or remove admissions.
assert len(cohort) == n_before_age_merge, \
    "The age merge changed the admission count."

# Estimate age in the year of each admission.
cohort["age_at_admission"] = (
    cohort["anchor_age"]
    + cohort["admittime"].dt.year
    - cohort["anchor_year"]
)

# Check missing ages before applying the adult criterion.
missing_age_count = int(cohort["age_at_admission"].isna().sum())
under_18_count = int(cohort["age_at_admission"].lt(18).sum())

print("Admissions before age eligibility:", n_before_age_merge)
print("Admissions with missing age:", missing_age_count)
print("Admissions with estimated age below 18:", under_18_count)
print("\nEstimated age at admission:")
print(cohort["age_at_admission"].describe())



Admissions before age eligibility: 528551
Admissions with missing age: 0
Admissions with estimated age below 18: 0

Estimated age at admission:
count    528551.000000
mean         58.743147
std          19.128298
min          18.000000
25%          45.000000
50%          60.000000
75%          73.000000
max         106.000000
Name: age_at_admission, dtype: float64


### Interpretation — Age Validation

The patient-information merge preserved all 528,551 admissions.
Every admission had the information needed to estimate age, and none
had an estimated age below 18 years.

The median estimated age was 60 years, with the middle 50% of admissions
between 45 and 73 years. These statistics describe admissions, so patients
with multiple admissions contribute more than once.

Applying the adult eligibility criterion therefore removed no admissions.
Age is a year-based estimate and should not be interpreted as exact age.

In [ ]:
# Stop if age information is missing rather than silently excluding records.
if cohort["age_at_admission"].isna().any():
    raise ValueError("Investigate missing ages before applying eligibility.")

# Record the cohort size before applying the adult criterion.
n_before_age = len(cohort)

# Retain admissions with estimated age of at least 18 years.
cohort = cohort.loc[cohort["age_at_admission"].ge(18)].copy()

# Save counts for the cohort-flow summary.
n_after_age = len(cohort)
age_removed = n_before_age - n_after_age

print("Admissions excluded for age below 18:", age_removed)
print("Admissions remaining:", n_after_age)

Admissions excluded for age below 18: 0
Admissions remaining: 528551


## 2.6 — Review Admission Types

### What we are checking

We summarize the recorded admission types and check for missing values
among admissions remaining after age eligibility.

### Why this matters

We inspect the exact categories before implementing an admission-type
exclusion. Excluding an elective admission from the prediction cohort
is separate from deciding whether a future planned admission counts
as a readmission outcome.

This cell summarizes the data without removing admissions.

In [ ]:
# Standardize admission-type descriptions without changing the source column.
admission_type_clean = (
    cohort["admission_type"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Count admissions in each category, including missing values.
admission_type_summary = (
    admission_type_clean
    .value_counts(dropna=False)
    .rename_axis("admission_type")
    .to_frame("admissions")
)

# Calculate each category's percentage of the current cohort.
admission_type_summary["percent"] = (
    admission_type_summary["admissions"] / len(cohort) * 100
).round(2)

print("Admissions reviewed:", len(cohort))
display(admission_type_summary)

Admissions reviewed: 528551


,admissions,percent
admission_type,,
EW EMER.,167884,31.76
EU OBSERVATION,119286,22.57
OBSERVATION ADMIT,80810,15.29
URGENT,51912,9.82
SURGICAL SAME DAY ADMISSION,42724,8.08
DIRECT OBSERVATION,24511,4.64
DIRECT EMER.,21212,4.01
ELECTIVE,13019,2.46
AMBULATORY OBSERVATION,7193,1.36


### Interpretation — Admission-Type Review

Nine admission types were recorded among 528,551 admissions, with no
missing admission-type values.

The 13,019 admissions classified as `ELECTIVE` (2.46%) were retained.
Our target is any qualifying recorded readmission within 30 days,
regardless of whether the starting admission was scheduled.

Admission type was reviewed to describe the cohort, not to determine
eligibility. No admissions were excluded at this stage.

The cohort remained at 528,551 admissions.

## 2.7 — Cohort Flow Summary

We summarize the exclusions applied sequentially and verify that the
remaining admission count matches the working cohort. Each excluded
admission is counted only once in this flow.

In [ ]:
# Summarize the previously recorded counts; do not filter again.
cohort_flow = pd.DataFrame({
    "stage": [
        "Initial admissions",
        "After usable timestamp requirement",
        "After time-order exclusion",
        "After death/hospice exclusion",
        "After adult eligibility",
        "After admission-type review"
    ],
    "admissions_remaining": [
        n_initial,
        n_after_time,
        n_after_time_order,
        n_after_death_hospice,
        n_after_age,
        len(cohort)
    ]
})

# Calculate exclusions relative to the preceding stage.
cohort_flow["excluded_at_stage"] = (
    cohort_flow["admissions_remaining"]
    .shift(1)
    .sub(cohort_flow["admissions_remaining"])
    .fillna(0)
    .astype(int)
)

display(cohort_flow)

# Report people separately from admissions.
print("Current admissions:", len(cohort))
print("Current unique patients:", cohort["subject_id"].nunique())

,stage,admissions_remaining,excluded_at_stage
0,Initial admissions,546028,0
1,After usable timestamp requirement,546028,0
2,After time-order exclusion,545853,175
3,After death/hospice exclusion,528551,17302
4,After adult eligibility,528551,0
5,After admission-type review,528551,0


Current admissions: 528551
Current unique patients: 216762


### Interpretation — Cohort Flow

Starting with 546,028 admissions, we excluded 175 with reversed
admission/discharge timestamps and 17,302 meeting death or hospice criteria.

No admissions were excluded for unusable timestamps or age below 18.
All admission types, including elective admissions, were retained.

The resulting Step 2 cohort contained **528,551 admissions from
216,762 unique patients**.

Exclusions were applied sequentially, so each removed admission was
counted once. This cohort defines potential prediction observations;
readmission labels and outcome eligibility will be addressed in Step 3.

In [ ]:
# Confirm that every prediction observation has valid identifiers.
assert cohort[["subject_id", "hadm_id"]].notna().all().all(), \
    "Missing patient or admission identifiers."

# Confirm that each admission appears once.
assert cohort["hadm_id"].is_unique, \
    "Duplicate admission identifiers found."

# Confirm that timestamps remain usable and correctly ordered.
assert cohort[["admittime", "dischtime"]].notna().all().all(), \
    "Missing admission or discharge timestamps."

assert cohort["dischtime"].ge(cohort["admittime"]).all(), \
    "Reversed timestamps remain."

# Confirm adult eligibility.
assert cohort["age_at_admission"].notna().all(), \
    "Missing estimated ages."

assert cohort["age_at_admission"].ge(18).all(), \
    "Admissions below age 18 remain."

# Recheck discharge criteria using the current filtered cohort.
final_disposition = (
    cohort["discharge_location"]
    .astype("string")
    .str.strip()
    .str.upper()
)

assert not cohort["hospital_expire_flag"].eq(1).any(), \
    "Admissions flagged for in-hospital death remain."

assert not final_disposition.eq("DIED").fillna(False).any(), \
    "Admissions with discharge location DIED remain."

assert not final_disposition.str.contains("HOSPICE", na=False).any(), \
    "Hospice discharges remain."

# Verify that sequential exclusion counts reconcile with the cohort size.
assert n_initial - (
    time_removed + time_order_removed + death_hospice_removed + age_removed
) == len(cohort), "Exclusion counts do not reconcile."

print("All final cohort checks passed.")
print("Admissions:", len(cohort))
print("Unique patients:", cohort["subject_id"].nunique())

All final cohort checks passed.
Admissions: 528551
Unique patients: 216762


## 2.8 — Final Cohort Validation

We verify that admission identifiers are unique, required identifiers
and timestamps are present, discharge does not precede admission,
and all retained admissions meet the adult and death/hospice criteria.

We also confirm that sequential exclusion counts reconcile with the
remaining cohort size. These checks do not modify the cohort.

## 2.9 — Step 2 Summary and Export

### Cohort results

- Initial admissions: **546,028**
- Excluded for unusable timestamps: **0**
- Excluded for discharge preceding admission: **175**
- Excluded for death or hospice: **17,302**, counting overlaps once
- Excluded for age below 18: **0**
- Excluded based on admission type: **0**
- Step 2 cohort: **528,551 admissions from 216,762 patients**

### Decisions to remember

- Each row represents one potential index admission.
- Patients may contribute multiple admissions.
- Equal admission/discharge timestamps were retained.
- Death was identified using either mortality indicator.
- Elective admissions were retained.
- Original source values were preserved.

### Handoff to Step 3

Step 2 defines potential prediction observations, not the final labeled
modeling sample.

Step 3 will define qualifying readmissions, examine overlapping encounters
and follow-up limitations, and document any additional outcome-related
exclusions.

Future admissions must be searched in the appropriate source admission
history, not only among admissions eligible as prediction observations.

Later patient splits must keep each patient's admissions together.
Feature merges must preserve admission identifiers and avoid unintended
row loss or duplication.

In [ ]:
# Count eligible admissions contributed by each patient.
cohort_admits_per_patient = (
    cohort.groupby("subject_id")["hadm_id"].nunique()
)

print("Admissions per patient in the Step 2 cohort:")
print(cohort_admits_per_patient.describe())

# Compare columns with the original admissions table.
# This reports actual changes rather than relying on old handwritten notes.
print("\nColumns added:")
print(sorted(set(cohort.columns) - set(admissions.columns)))

print("\nOriginal columns absent from cohort:")
print(sorted(set(admissions.columns) - set(cohort.columns)))

print("\nCohort dimensions:", cohort.shape)

Admissions per patient in the Step 2 cohort:
count    216762.000000
mean          2.438393
std           3.576213
min           1.000000
25%           1.000000
50%           1.000000
75%           3.000000
max         237.000000
Name: hadm_id, dtype: float64

Columns added:
['age_at_admission', 'anchor_age', 'anchor_year']

Original columns absent from cohort:
[]

Cohort dimensions: (528551, 19)


In [ ]:
# Create the configured output directory if needed.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save the validated cohort for Step 3.
# Running this cell replaces the existing file at this path.
COHORT_FILE = PROCESSED_DIR / "cohort_step2.csv"
cohort.to_csv(COHORT_FILE, index=False)

# Save the aggregate cohort-flow table for convenient review.
FLOW_FILE = PROCESSED_DIR / "cohort_step2_flow.csv"
cohort_flow.to_csv(FLOW_FILE, index=False)

# Read back identifiers to verify the saved admission and patient counts.
saved_ids = pd.read_csv(
    COHORT_FILE,
    usecols=["subject_id", "hadm_id"]
)

assert len(saved_ids) == len(cohort), "Saved row count differs."
assert saved_ids["hadm_id"].is_unique, "Saved admission IDs repeat."
assert set(saved_ids["hadm_id"]) == set(cohort["hadm_id"]), \
    "Saved admission identifiers differ."
assert saved_ids["subject_id"].nunique() == cohort["subject_id"].nunique(), \
    "Saved patient count differs."

print("Cohort saved:", COHORT_FILE)
print("Cohort flow saved:", FLOW_FILE)
print("Verified admissions:", len(saved_ids))
print("Verified patients:", saved_ids["subject_id"].nunique())

In [ ]:
# Show the exact columns that will be exported.
print("Number of columns:", cohort.shape[1])

for column in cohort.columns:
    print(column)

Number of columns: 19
subject_id
hadm_id
admittime
dischtime
deathtime
admission_type
admit_provider_id
admission_location
discharge_location
insurance
language
marital_status
race
edregtime
edouttime
hospital_expire_flag
anchor_age
anchor_year
age_at_admission
